In [1]:
import pandas as pd
import os
import requests
from tqdm import tqdm

base_dir = "/home/rahul/code/gsoc/Evaluation-Test-ArtExtract/Task2/nga_dataset"
img_dir = os.path.join(base_dir, "images")
os.makedirs(img_dir, exist_ok=True)

print("Downloading NGA Metadata CSVs directly from GitHub...")

objects_url = "https://raw.githubusercontent.com/NationalGalleryOfArt/opendata/main/data/objects.csv"
images_url = "https://raw.githubusercontent.com/NationalGalleryOfArt/opendata/main/data/published_images.csv"

df_objects = pd.read_csv(objects_url, low_memory=False)
df_images = pd.read_csv(images_url, low_memory=False)


print("Filtering for Paintings...")
# only need paintings 
df_paintings = df_objects[df_objects['classification'].astype(str).str.lower() == 'painting']

merged_df = pd.merge(
    df_paintings, 
    df_images, 
    left_on='objectid', 
    right_on='depictstmsobjectid', 
    how='inner'
)

# Droping rows where the IIIF URL is missing
merged_df = merged_df.dropna(subset=['iiifurl'])

print(f"Found {len(merged_df)} total paintings with images.")

DOWNLOAD_LIMIT = 4053
df_to_download = merged_df.head(DOWNLOAD_LIMIT).copy()

print(f"Starting download of {len(df_to_download)} images...")

successful_downloads = []

for index, row in tqdm(df_to_download.iterrows(), total=len(df_to_download)):
    obj_id = row['objectid']
    base_iiif_url = row['iiifurl']
    
    # IIIF : /full/!512,512/0/default.jpg tells the server to resize it 
    img_url = f"{base_iiif_url}/full/!512,512/0/default.jpg"
    
    save_path = os.path.join(img_dir, f"{obj_id}.jpg")
    
    # Skip if already downloaded 
    if not os.path.exists(save_path):
        try:
            response = requests.get(img_url, timeout=10)
            if response.status_code == 200:
                with open(save_path, 'wb') as f:
                    f.write(response.content)
                successful_downloads.append(save_path)
            else:
                pass # Silently skip broken links
        except Exception as e:
            pass # Silently skip connection timeouts

print(f"Finished! Successfully downloaded {len(successful_downloads)} new images.")

df_to_download.to_csv(os.path.join(base_dir, "nga_paintings_metadata.csv"), index=False)

Filtering for Paintings...
Found 4053 total paintings with images.
Starting download of 4053 images...


100%|██████████| 4053/4053 [53:03<00:00,  1.27it/s]  

Finished! Successfully downloaded 2041 new images.
